# 作品① · 抖音数据完整分析

目标：用 pandas 分析自己 17 条作品，回答三个问题——

1. 我平时的真实水平是多少（不被爆款骗）？
2. 哪类选题、哪种数据表现更好？
3. 涨粉靠的是什么？

用法：从上到下依次运行每个代码格。先看输出，再看格子上面的中文说明。

## 第 1 步：读数据，看前 5 行

`read_excel` 读表格，`head()` 看前 5 行。观察每列是什么。

In [ ]:
import pandas as pd

df = pd.read_excel('C:/Users/Administrator/Desktop/作品列表导出.xlsx')
print('数据大小：', df.shape)
df.head()

## 第 2 步：认识数据

17 行作品，19 列字段。运行 `df.info()` 看每列类型和空值。

注意 topic 列已经填好（情感关系/自我成长/大学生活/情绪日常 四类），hook 和 tags 暂时空着。

In [ ]:
print('行数', df.shape[0], '列数', df.shape[1])
df[['title', 'views', 'finish_rate', 'topic']].head(10)

## 第 3 步：清洗数据（很重要）

两个坑必须先处理：

1. **6 条「自见」作品播放量是 0**，是自己可见的，要排除，只留「公开」。
2. `cover_ctr` 列有个 `-` 是无效值，`to_numeric(errors='coerce')` 把它变空值。

同时算好**互动率** =（赞+分享+评论+收藏）÷ 播放量，后面反复用。

In [ ]:
print('清洗前：')
print(df['audit_status'].value_counts())

df = df[df['audit_status'] == '公开'].copy()
df['cover_ctr'] = pd.to_numeric(df['cover_ctr'], errors='coerce')
df['互动率'] = (df['likes'] + df['shares'] + df['comments'] + df['favs']) / df['views']

print('清洗后剩', len(df), '条公开作品')

## 第 4 步：真实水平——平均数 vs 中位数

有一条 298 万播放的爆款，会把「平均播放量」拉得极高。

判断平时真实水平要看**中位数**。运行看看两者差多少倍。

In [ ]:
print('平均播放量：', round(df['views'].mean()))
print('中位数播放量：', round(df['views'].median()))

df.sort_values('views', ascending=False)[['title', 'topic', 'views', 'finish_rate']].head()

## 第 5 步：按选题（topic）分组——回答「哪类选题最好」

`groupby('topic')` 按选题类型分组，看每类的条数、平均播放、中位播放、平均互动率、平均完播。

**先别急着下结论**——盯着「大学生活」那一组看，它的平均值正常吗？下一步会揭晓。

In [ ]:
result = df.groupby('topic').agg(
    条数=('views', 'size'),
    平均播放=('views', 'mean'),
    中位播放=('views', 'median'),
    平均互动率=('互动率', 'mean'),
    平均完播=('finish_rate', 'mean')
)
result.round(3)

## 第 6 步：识破爆款制造的假象（关键）

「大学生活」组只有 1 条，就是那条 298 万爆款，所以它一组的平均播放是 298 万——
**这不是这个选题的普遍水平，是那一条的独角戏**。

做法：把爆款（播放量 > 10 万）暂时剔除，再看各选题的真实表现。

这就是教程反复强调的：**算法和爆款是最大的混杂变量，别用绝对播放量下结论**。

In [ ]:
no_hit = df[df['views'] < 100000]
print('剔除爆款后剩', len(no_hit), '条')

no_hit.groupby('topic').agg(
    条数=('views', 'size'),
    中位播放=('views', 'median'),
    平均互动率=('互动率', 'mean')
).round(3)

## 第 7 步：涨粉靠什么

播放量高不等于涨粉多。看看涨粉排名，是不是全靠那一条爆款。

In [ ]:
df[['title', 'topic', 'views', 'new_followers']].sort_values('new_followers', ascending=False).head()

## 第 8 步：我的观察（这是你的成果，可直接填回计划书）

下面这段是基于真实数据的结论。把它复制进 Notion「作品①计划书」的第三部分，计划书就活了。

---

**1. 数据概况**
17 条作品里有 6 条是「自见」（播放量为 0），排除后真正能分析的是 **11 条公开作品**。

**2. 真实水平要看中位数，别被爆款骗**
「#大学生」那条播放量 **2,981,390**（约 298 万），是第二名的 293 倍。平均播放 **274,035**，中位数只有 **1,112**，差 246 倍。判断平时水平要看中位数 1,112。

**3. 哪类选题最好？——这个问题差点被爆款带偏**
- 直接按 topic 分组：「大学生活」组平均播放 298 万，看着最强，但**这组只有 1 条，就是那条爆款**，是假象。
- 剔除爆款后：**情感关系**（5 条）中位播放 5,805 最强；**自我成长**（3 条）中位播放 543 最弱，但它的平均互动率 5.9% 最高，情绪日常 5.0%，情感关系 3.2%。
- 结论：**情感关系类更能拉播放，自我成长类互动更用心但看得人少**。这就是「用率不用绝对数」的意义。

**4. 涨粉几乎全靠那一条爆款**
「#大学生」一条带来 **251 个新粉丝**，其余作品涨粉最多 1 个。日常内容涨粉能力很弱。

**5. 假设验证情况**
- 假设①（选题影响播放）：**已验证**——情感关系播放最强，自我成长最弱但互动率最高。
- 假设②（钩子影响完播）：**还没法验证**——hook 列空着，需看视频手工补。
- 假设③（收藏率与涨粉）：**部分验证**——涨粉高度集中在爆款，样本不足以证明是收藏驱动。

**一句话结论**：下一条作品用「情感关系」选题拉播放，同时研究那条 298 万爆款的开头钩子，试着把自我成长类的互动率优势转化成播放量。

---

**已知局限**：16 条图文 + 1 条视频，结论主要适用于图文；6 条自见已排除；样本量小，下周同一时间再导出、攒够 8 周才能下稳定结论。